# Líneas de base BLEU / chrF sobre `v2-nostrat`

Para cada split de `v2-nostrat`:

1. **descarga un modelo** de traducción y **genera las hipótesis** (traducciones),
2. calcula, a nivel de corpus, **BLEU** y **chrF** (con sus variantes) comparando las
   hipótesis contra la referencia, usando `sacrebleu`.

Pensado para alimentar la tabla de líneas de base del póster.

## Modelo

Usa los **checkpoints fine-tuneados** de `v2-nostrat`, uno por dirección (definidos en
`MODEL_QOM2ES` / `MODEL_ES2QOM`). En `qom-mt` esos modelos se guardaron en el
working dir, así que tenés que apuntarlos a la **ruta local**
donde los tengas o importarlos del Hub. Como línea de
base opcional está `BASE_MODEL` (NLLB sin fine-tuning). La inferencia replica la del
pipeline de `qom-mt`: tokenizer NLLB, `forced_bos_token_id` del idioma destino,
`num_beams=4`, `no_repeat_ngram_size=3`, `max_new_tokens=128`.

> Códigos de idioma NLLB: español = `spa_Latn`. Para qom se usa `grn_Latn` (guaraní) como
> **proxy**, igual que en `qom-mt` — NLLB no tiene qom, así que se aprovecha un código
> cercano.

## Dirección

`RUNS` es una lista de `(direction, model)`. `direction` es `"qom2es"` o `"es2qom"` y
define qué columna es la entrada y cuál la referencia. Los checkpoints fine-tuneados
suelen ser **por dirección**, por eso cada run lleva su propio modelo.

## Variantes de chrF

Se barre la grilla `word_order ∈ {0, 1, 2}` × `beta ∈ {1, 2}` con
`sklearn.model_selection.ParameterGrid` (la primitiva estándar para enumerar una grilla;
`GridSearchCV` no aplica: no hay un estimador con `fit`/validación cruzada, solo
evaluación directa de una métrica).


In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
import torch
import sacrebleu
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from sklearn.model_selection import ParameterGrid
# NllbTokenizer (slow, SentencePiece) — misma clase que usa qom-mt, para replicabilidad.
# (AutoTokenizer devolvería el fast, que puede tokenizar distinto el token de idioma.)
from transformers import AutoModelForSeq2SeqLM, NllbTokenizer

# Dataset empaquetado para Hugging Face (carpeta qomL-hf de master-thesis-corpus).
DATASET = "../master-thesis-corpus/data/out/qomL-hf"
CONFIG  = "v2-nostrat"

# Checkpoints fine-tuneados de v2-nostrat, uno por dirección.
MODEL_QOM2ES = "models/qom-mt-v2-nostrat/qom-mt-v2-nostrat-qom2es"
MODEL_ES2QOM = "models/qom-mt-v2-nostrat/qom-mt-v2-nostrat-es2qom"

# Línea de base opcional: NLLB sin fine-tuning.
BASE_MODEL = "facebook/nllb-200-distilled-600M"

# Runs a evaluar: (direction, model). Ambas direcciones con el modelo fine-tuneado.
RUNS = [
    ("qom2es", MODEL_QOM2ES),
    ("es2qom", MODEL_ES2QOM),
]

# Chequeo: no dejar los placeholders sin completar (evita un error críptico de HF).
for _dir, _m in RUNS:
    if "RUTA/O/ID" in _m:
        raise ValueError(
            f"Falta el checkpoint para '{_dir}': reemplazá MODEL_{_dir.upper()} por la "
            "ruta local o el id del Hub de tu modelo fine-tuneado (o usá BASE_MODEL "
            "para evaluar la línea de base NLLB)."
        )

# Códigos de idioma NLLB (qom vía proxy grn_Latn, como en qom-mt)
LANG = {"qom": "grn_Latn", "es": "spa_Latn"}
# direction -> (columna fuente, columna referencia)
DIRECTION_COLS = {"qom2es": ("qom", "es"), "es2qom": ("es", "qom")}

# Generación
MAX_LEN   = 128
NUM_BEAMS = 4
BATCH     = 8

# Grilla de variantes de chrF
CHRF_GRID = {"word_order": [0, 1, 2], "char_order": [
    4,5,6], "beta": [1, 2]}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DATASET : {DATASET}  (config: {CONFIG})")
print(f"RUNS    : {RUNS}")
print(f"DEVICE  : {DEVICE}")


In [ ]:
# ── Carga del dataset ─────────────────────────────────────────────────────────
ds = load_dataset(DATASET, CONFIG)
print(ds)


In [ ]:
# ── Modelo e inferencia (replica el pipeline NLLB de qom-mt) ──────────────────
def load_model(model_name: str):
    # NllbTokenizer + AutoModelForSeq2SeqLM: mismo patrón de carga que qom-mt.
    tokenizer = NllbTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()
    return model, tokenizer


def translate(model, tokenizer, texts, src_lang, tgt_lang) -> list[str]:
    tokenizer.src_lang = src_lang
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    preds = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i + BATCH]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_LEN).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                 max_new_tokens=MAX_LEN, num_beams=NUM_BEAMS,
                                 no_repeat_ngram_size=3)
        preds += tokenizer.batch_decode(out, skip_special_tokens=True)
    return preds


In [ ]:
# ── Métricas (BLEU + grilla de chrF, a nivel de corpus) ───────────────────────
CHRF_LABEL = {0: "chrF", 1: "chrF+", 2: "chrF++"}

def chrf_name(word_order: int, beta: int) -> str:
    return f"{CHRF_LABEL[word_order]} (β={beta})"


def score_corpus(hyps, refs) -> list[dict]:
    rows = [{
        "metric": "BLEU", "word_order": None, "beta": None,
        "score": sacrebleu.corpus_bleu(hyps, [refs]).score,
    }]
    for params in ParameterGrid(CHRF_GRID):
        rows.append({
            "metric": chrf_name(params["word_order"], params["beta"]),
            "word_order": params["word_order"],
            "char_order": params["char_order"],
            "beta": params["beta"],
            "score": sacrebleu.corpus_chrf(hyps, [refs], **params).score,
        })
    return rows


In [ ]:
# ── Inferencia + evaluación por run y split ───────────────────────────────────
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

records = []
for direction, model_name in RUNS:
    src_col, ref_col = DIRECTION_COLS[direction]
    src_lang, tgt_lang = LANG[src_col], LANG[ref_col]
    print(f"\n=== {direction}  ({src_col} -> {ref_col})  |  {model_name} ===")

    model, tokenizer = load_model(model_name)

    for split_name, split in ds.items():
        sources = split[src_col]
        refs    = split[ref_col]
        print(f"  [{split_name}] traduciendo {len(sources)} oraciones ...")
        hyps = translate(model, tokenizer, sources, src_lang, tgt_lang)

        # guardo las predicciones para inspección / reuso
        pd.DataFrame({"id": split["id"], "source": sources,
                      "reference": refs, "prediction": hyps}).to_csv(
            out_dir / f"preds_{direction}_{split_name}.csv", index=False)

        for row in score_corpus(hyps, refs):
            records.append({"direction": direction, "model": model_name,
                            "split": split_name, "n": len(sources), **row})

    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame.from_records(records)
results["score"] = results["score"].round(2)
results


In [ ]:
# ── Tabla resumen por dirección (métricas × splits) ───────────────────────────
metric_order = ["BLEU"] + [chrf_name(wo, b) for wo in (0, 1, 2) for b in (1, 2)]
split_order = [s for s in ["train", "validation", "test"] if s in set(results["split"])]

for direction in results["direction"].unique():
    sub = results[results["direction"] == direction]
    table = (sub.pivot(index="metric", columns="split", values="score")
                .reindex(index=metric_order, columns=split_order))
    print(f"\n### {direction}")
    print(table.to_string())
    table.to_csv(Path("results") / f"metricas_{CONFIG}_{direction}.csv")


## Notas

- Los números son los del **modelo fine-tuneado** que apuntes en `MODEL_QOM2ES` /
  `MODEL_ES2QOM`. Si querés la línea de base NLLB vainilla, usá `BASE_MODEL` en `RUNS`.
  Recordá que los checkpoints de `qom-mt` viven en el output de Kaggle, no en el Hub.
- La inferencia corre en GPU si hay (`DEVICE`); en CPU funciona pero es lenta (beam search
  sobre todos los splits). Para una prueba rápida, restringí `ds` a un split o a un subset.
- `word_order`: `0` = chrF, `1` = chrF+, `2` = chrF++ (agrega n-gramas de palabra).
  `beta`: peso de *recall* vs *precision* (`beta=2` prioriza recall, default de sacrebleu).
- Todo se calcula a nivel de **corpus**. Las predicciones quedan en
  `results/preds_<direction>_<split>.csv` y las tablas en
  `results/metricas_<config>_<direction>.csv`.
